# Gold-Business logic aggregates

## Delay analysis — avg delay by carrier and by origin airport

In [0]:
from pyspark.sql import functions as F

df_silver = spark.read.table("flightdata.silver.flightdata")

# Avg delay by carrier
delay_by_carrier = df_silver.groupBy("carrier").agg(
    F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
    F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
    F.count("*").alias("flight_count")
).orderBy(F.desc("avg_dep_delay"))

# Avg delay by origin airport
delay_by_origin = df_silver.groupBy("origin").agg(
    F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
    F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
    F.count("*").alias("flight_count")
).orderBy(F.desc("avg_dep_delay"))

delay_by_carrier.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.delay_by_carrier")

delay_by_origin.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.delay_by_origin")

## Flight volume trends — by route, by date, by carrier

In [0]:
# Volume by route (origin -> dest)
volume_by_route = df_silver.groupBy("origin", "dest").agg(
    F.count("*").alias("flight_count")
).orderBy(F.desc("flight_count"))

# Volume by date
volume_by_date = df_silver.groupBy("date").agg(
    F.count("*").alias("flight_count")
).orderBy("date")

# Volume by carrier
volume_by_carrier = df_silver.groupBy("carrier").agg(
    F.count("*").alias("flight_count")
).orderBy(F.desc("flight_count"))

volume_by_route.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.volume_by_route")

volume_by_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.volume_by_date")

volume_by_carrier.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.volume_by_carrier")

## Airport/carrier performance summary — combined view

In [0]:
performance_summary = df_silver.groupBy("carrier", "origin").agg(
    F.count("*").alias("total_flights"),
    F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
    F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
    F.round(F.avg("air_time"), 2).alias("avg_air_time"),
    F.round((F.sum(F.when(F.col("arr_delay") <= 0, 1).otherwise(0)) / F.count("*")) * 100, 2)
        .alias("on_time_pct")
).orderBy(F.desc("total_flights"))

performance_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("flightdata.gold.performance_summary")

In [0]:
display(spark.read.table("flightdata.gold.delay_by_carrier"))
display(spark.read.table("flightdata.gold.volume_by_date"))
display(spark.read.table("flightdata.gold.performance_summary"))